In [1]:

import json
from datetime import datetime

DATA_FILE = "bank.json"


def load():
    try:
        return json.load(open(DATA_FILE))
    except FileNotFoundError:
        return {}


def save(accounts):
    json.dump(accounts, open(DATA_FILE, "w"), indent=2)


def create_account(accounts, name):
    acc_id = f"ACC-{len(accounts) + 1:04d}"
    accounts[acc_id] = {"name": name, "balance": 0.0, "history": []}
    return acc_id


def move_money(accounts, acc_id, amount, kind, other=None):
    acc = accounts[acc_id]
    if kind in ("deposit", "transfer in"):
        acc["balance"] += amount
    else:
        if amount > acc["balance"]:
            raise ValueError("Insufficient funds")
        acc["balance"] -= amount
    acc["history"].append(f"{datetime.now():%Y-%m-%d %H:%M}  {kind}  ${amount:.2f}"
                          + (f"  ({other})" if other else ""))


def main():
    accounts = load()

    while True:
        print("\n1. Create  2. Deposit  3. Withdraw  4. Transfer  5. Balance  6. History  0. Exit")
        choice = input("> ").strip()

        try:
            if choice == "1":
                acc_id = create_account(accounts, input("Name: ").strip())
                print(f"Created: {acc_id}")
            elif choice in "2356":
                acc_id = input("Account ID: ").strip()
                if choice == "5":
                    print(f"Balance: ${accounts[acc_id]['balance']:.2f}")
                elif choice == "6":
                    print("\n".join(accounts[acc_id]["history"]) or "No transactions")
                else:
                    amount = float(input("Amount: "))
                    if amount <= 0:
                        raise ValueError("Amount must be positive")
                    move_money(accounts, acc_id, amount, "deposit" if choice == "2" else "withdraw")
                    print("Done.")
            elif choice == "4":
                src, dst = input("From: ").strip(), input("To: ").strip()
                amount = float(input("Amount: "))
                move_money(accounts, src, amount, "transfer out", f"to {dst}")
                move_money(accounts, dst, amount, "transfer in", f"from {src}")
                print("Done.")
            elif choice == "0":
                break
            else:
                continue
            save(accounts)
        except (ValueError, KeyError) as e:
            print(f"Error: {e}")


if __name__ == "__main__":
    main()


1. Create  2. Deposit  3. Withdraw  4. Transfer  5. Balance  6. History  0. Exit


KeyboardInterrupt: Interrupted by user